# osss quickstart
Index an S3 prefix or a local folder with OpenSearch and search it, all inside this notebook.

In [ ]:
%pip install -q "osss @ git+https://github.com/utkarsh5026/osss.git"

## Option A: the app
Everything below can also be done with clicks.

In [ ]:
import osss
osss.ui()

## Option B: Python
Start a local OpenSearch. The first run downloads it (~250 MB); later runs reconnect instantly, even after a kernel restart.

In [ ]:
cluster = osss.start()
cluster

Preview a source before indexing it:

In [ ]:
SOURCE = "s3://my-bucket/docs/"   # or a local folder, e.g. "./docs"
cluster.scan(SOURCE)

In [ ]:
idx = cluster.create_index(
    "docs", SOURCE,
    include=["*.pdf", "*.md", "*.docx", "*.csv"],
    chunk_size=800, chunk_overlap=120,
    analyzer="english",
    recreate=True,
)
idx.last_report

In [ ]:
idx.search("what is the refund policy?")

In [ ]:
res = idx.search("refund", file_type="pdf", fuzzy=True, k=20)
res.to_pandas().head()

## Semantic & hybrid search
Needs `pip install 'osss[semantic]'` for local models, or use a Bedrock model id such as `amazon.titan-embed-text-v2:0`.

In [ ]:
sem = cluster.create_index("docs-semantic", SOURCE, semantic=True, recreate=True)
sem.search("can I get my money back after a month?")   # hybrid by default

## Clean up
The server keeps running in the background (your indexes are kept on disk). Stop it when you're done:

In [ ]:
# osss.stop()